# **0. 준비**

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score

In [12]:
# 시각화에 사용할 팔레트
plt.style.available

sns.color_palette("pastel")
sns.set(palette='pastel')

In [13]:
# !git clone https://github.com/CheckNanbang/2025-weather-data-contest.git

In [15]:
%cd /content/drive/MyDrive/체크난방/code/final/

/content/drive/.shortcut-targets-by-id/1enonykTnS-tycW5dfJso6oq4hJtGXU4X/체크난방/code/final


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 23.7 MB/s eta 0:00:00


In [16]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import optuna

from preproccesor import WeatherDataPreprocessor
warnings.filterwarnings(action='ignore', category=UserWarning)

In [ ]:
!nvidia-smi

Fri Jun 27 04:35:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **1. 데이터 불러오기**

#### (1) train data

In [ ]:
df_train = pd.read_csv('/content/drive/MyDrive/체크난방/data/weather_data_imputed2.csv')
df_train.head(3)

,tm,branch_id,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi,...,date,year,month,day,hour,quarter,day_of_week,season,is_weekend,hour_group
0,2021-01-01 01:00:00,A,-10.1,78.3,0.5,0.0,0.0,68.2,0.0,-8.2,...,2021-01-01,2021,1,1,1,1,4,Winter,Weekday,0
1,2021-01-01 02:00:00,A,-10.2,71.9,0.6,0.0,0.0,69.9,0.0,-8.6,...,2021-01-01,2021,1,1,2,1,4,Winter,Weekday,0
2,2021-01-01 03:00:00,A,-10.0,360.0,0.0,0.0,0.0,69.2,0.0,-8.8,...,2021-01-01,2021,1,1,3,1,4,Winter,Weekday,1


In [ ]:
preprocessor = WeatherDataPreprocessor()

cluster_results = preprocessor.preprocess_data(df_train, is_test=False)
cluster0_df = cluster_results.get('cluster0')
# cluster1_df = cluster_results.get('cluster1')
cluster1_summer_df = cluster_results.get('cluster1_summer')
cluster1_non_summmer_df = cluster_results.get('cluster1_non_summer')
cluster2_summer_df = cluster_results.get('cluster2_summer')
cluster2_non_summmer_df = cluster_results.get('cluster2_non_summer')
cluster3_df = cluster_results.get('cluster3')

데이터 전처리 시작...
cluster2_summer 전처리 중...
cluster2_non_summer 전처리 중...
cluster1_summer 전처리 중...
cluster1_non_summer 전처리 중...
cluster0 전처리 중...
cluster3 전처리 중...
클러스터별 데이터 전처리 완료!


#### (2) test data

In [ ]:
df_test = pd.read_csv('/content/drive/MyDrive/체크난방/data/non_null_test_data_dh.csv')
df_test.head(3)

,tm,branch_id,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi,...,date,year,month,day,hour,quarter,day_of_week,season,is_weekend,hour_group
0,2024-01-01 0:00,A,0.5,171.3,0.8,2.5,0.0,97.1,0.0,0.3,...,2024-01-01,2024,1,1,0,1,0,Winter,Weekday,0
1,2024-01-01 1:00,A,0.4,93.7,1.0,0.0,0.0,96.8,0.0,0.1,...,2024-01-01,2024,1,1,1,1,0,Winter,Weekday,0
2,2024-01-01 2:00,A,-0.1,133.0,0.8,0.0,0.0,97.0,0.0,0.0,...,2024-01-01,2024,1,1,2,1,0,Winter,Weekday,0


In [ ]:
df_test["tm"] = pd.to_datetime(df_test["tm"]).dt.strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
preprocessor = WeatherDataPreprocessor()

cluster_results = preprocessor.preprocess_data(df_test, is_test=True)
cluster0_df_test = cluster_results.get('cluster0')
# cluster1_df_test = cluster_results.get('cluster1')
cluster1_summer_df_test = cluster_results.get('cluster1_summer')
cluster1_non_summer_df_test = cluster_results.get('cluster1_non_summer')
cluster2_summer_df_test = cluster_results.get('cluster2_summer')
cluster2_non_summer_df_test = cluster_results.get('cluster2_non_summer')
cluster3_df_test = cluster_results.get('cluster3')

데이터 전처리 시작...
cluster2_summer 전처리 중...
cluster2_non_summer 전처리 중...
cluster1_summer 전처리 중...
cluster1_non_summer 전처리 중...
cluster0 전처리 중...
cluster3 전처리 중...
클러스터별 데이터 전처리 완료!


# **2. Cluster별 Best model**

#### (1) Cluster0

In [ ]:
# 데이터 준비
df_cluster0 = cluster0_df.copy()
df_cluster0 = df_cluster0.sort_values(["tm", "branch_id"]).reset_index(drop=True)

# 타겟 지정
target_col = "heat_demand"
y = df_cluster0[target_col]

# object 컬럼 원핫 인코딩
object_cols = df_cluster0.select_dtypes(include="object").columns.tolist()
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded_array = encoder.fit_transform(df_cluster0[object_cols])
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(object_cols))

# 나머지 수치형 변수
exclude_cols = object_cols + [target_col, "heat_demand", "heat_demand_log"]
numeric_df = df_cluster0.drop(columns=exclude_cols)

# 최종 feature
X = pd.concat([numeric_df.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)
X['tm'] = pd.to_datetime(X['tm'])  # 만약 이미 datetime이면 이 줄 생략

# 1시간 단위로 시간 블록 생성 (연-월-일-시까지 포함됨)
X['time_block'] = X['tm'].dt.strftime('%Y-%m-%d %H:00:00')

X = X.drop('tm',axis=1)

# 고유한 시간 블록 추출 및 정렬
time_blocks = np.sort(X['time_block'].unique())

In [ ]:
selected_features_995_nonlog =  ['season_group_winter','season_group_early_winter','ta_chi_roll24','ta_chi_roll168','branch_id_N','heating_degree','branch_id_K',
                                 'ta_chi','branch_id_Q','si_lag1','ta_chi_roll72','coldwave','DI_interaction','si_roll3','branch_id_E','hour','branch_id_J','branch_id_I',
                                 'si_lag9','ta_chi_lag1','branch_id_F','si_lag13','si_lag10','branch_id_O','si_lag17','si_lag15','ws_roll168','si_lag2','day_flag',
                                 'ta_roll168','weekofyear','ta','hour_sin','hour_of_week','ta_lag1','ta_roll72','hour_cos','si_lag26','season_group_summer','si_lag12','ta_diff_chi','year',
                                 'si_lag18','wd_cos_lag10','si_lag3','season_group_mid_season','si_lag16','month','ta_roll3','si_lag19','ta_lag24','ta_roll24','ta_zone_low','si_lag25',
                                 'ta_lag9','ta_zone_mid','si','ta_chi_roll3','si_lag23','hm_lag19','month_cos','ta_lag22','ta_lag12','si_lag11','ta_lag23','month_sin','ws','si_lag14','si_roll168',
                                 'cumulative_si','day','si_lag4','heatwave','ta_lag2','ta_lag25','ta_lag3','hm_roll168','si_lag5','ta_lag6','si_roll24','rn_hr1_roll168','ws_lag23','si_roll72',
                                 'ws_roll72','dew_point','monthday_sin','ws_roll24','hm_roll24','ta_lag10','ws_lag8','si_lag24','ta_lag4','rn_hr1_roll72','hm_roll72','weekhour_cos','ta_lag20',
                                 'monthday_cos','ta_lag7','ws_roll3','hm_lag7','wd_sin_lag1','ws_lag7','weekhour_sin','si_lag29','ws_lag22','hm_lag29','ta_chi_lag30','rn_hr1_roll24','ta_lag19',
                                 'rn_day','ws_lag5','ta_lag21','wd_cos_lag19','ws_lag24','wd_lag12','ta_zone_high','ws_lag18','hm_lag1','si_lag27','hm_lag17','rn_day_lag24','hm_lag2','ws_lag11',
                                 'wd_sin_lag12','ws_lag4','ws_lag21','hm_roll3','hm_lag8','hm_lag10','ta_lag8']

In [ ]:
# 4. 최종 모델 학습
X_selected_995_nonlog = X[selected_features_995_nonlog + ['time_block']]
X_selected_995_nonlog = X_selected_995_nonlog.drop('time_block',axis=1)

# 고정 Best Params
best_params = {
    'n_estimators': 930,
    'max_depth': 6,
    'learning_rate': 0.04610582490653898,
    'subsample': 0.5530677444010951,
    'colsample_bytree': 0.9372943847364046,
    'reg_alpha': 1.8929668699316567e-08,
    'reg_lambda': 1.9018409383453436e-08,
    'tree_method': 'gpu_hist',
    'predictor': 'gpu_predictor',
    'random_state': 42,
    'n_jobs': -1
}

sample_weight = np.log1p(y)  # 로그 가중치
final_model_cluster0 = XGBRegressor(**best_params)
final_model_cluster0.fit(X_selected_995_nonlog, y, sample_weight=sample_weight)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9372943847364046, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.04610582490653898, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=930, n_jobs=-1,
             num_parallel_tree=None, predictor='gpu_predictor', ...)

In [ ]:
# test 데이터 복사 및 전처리
df_test_cluster0 = cluster0_df_test.copy()
df_test_cluster0 = df_test_cluster0.sort_values(["tm", "branch_id"]).reset_index(drop=True)

# 타겟 지정
target_col = "heat_demand"
# y = df[target_col]

# object 컬럼 인코딩
encoded_array_test = encoder.transform(df_test_cluster0[object_cols])  # 학습된 encoder 사용
encoded_df_test = pd.DataFrame(encoded_array_test, columns=encoder.get_feature_names_out(object_cols))

# 수치형 + 인코딩 합치기
exclude_cols = object_cols + ['tm', target_col, "heat_demand", "heat_demand_log"]
numeric_df_test = df_test_cluster0.drop(columns=exclude_cols)
X_test = pd.concat([numeric_df_test.reset_index(drop=True), encoded_df_test.reset_index(drop=True)], axis=1)

# 선택된 변수로 축소
X_test_selected = X_test[selected_features_995_nonlog]

# 예측
y_pred_cluster0= final_model_cluster0.predict(X_test_selected)

# 결과를 test 데이터에 붙이기
df_test_cluster0["heat_demand_pred"] = y_pred_cluster0

# 출력
df_test_cluster0[["tm", "branch_id", "heat_demand_pred"]].head()

df_test_cluster0 = df_test_cluster0[["tm", "branch_id", "heat_demand_pred"]]

#### (2) Cluster1

In [ ]:
def calculate_sample_weights(y_values, weight_type='log1p'):
    """
    샘플 가중치 계산 함수
    """
    y_values = np.array(y_values)

    if weight_type == 'log1p':
        weights = np.log1p(y_values)
    elif weight_type == 'linear':
        weights = y_values
    elif weight_type == 'sqrt':
        weights = np.sqrt(np.maximum(y_values, 0))
    elif weight_type == 'quadratic':
        weights = np.square(y_values)
    else:
        raise ValueError("weight_type must be one of: 'log1p', 'linear', 'sqrt', 'quadratic'")

    # 가중치 정규화 (평균을 1로)
    weights = weights / np.mean(weights)
    return weights

def prepare_features(train_df, test_df, target_col, encoder=None):
    """피처 준비 및 인코딩"""
    # 범주형 변수 원핫 인코딩
    object_cols = train_df.select_dtypes(include="object").columns.tolist()

    if encoder is None:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        if object_cols:
            encoder.fit(train_df[object_cols])

    # 훈련 데이터 피처 준비
    if object_cols:
        X_train_obj = pd.DataFrame(
            encoder.transform(train_df[object_cols]),
            columns=encoder.get_feature_names_out(object_cols)
        )
        X_test_obj = pd.DataFrame(
            encoder.transform(test_df[object_cols]),
            columns=encoder.get_feature_names_out(object_cols)
        )
    else:
        X_train_obj = pd.DataFrame()
        X_test_obj = pd.DataFrame()

    # 수치형 변수
    exclude_cols = object_cols + [target_col, "tm", "heat_demand_log"]
    X_train_num = train_df.drop(columns=exclude_cols, errors='ignore')
    X_test_num = test_df.drop(columns=exclude_cols, errors='ignore')

    # 피처 결합
    X_train = pd.concat([X_train_num.reset_index(drop=True), X_train_obj.reset_index(drop=True)], axis=1)
    X_test = pd.concat([X_test_num.reset_index(drop=True), X_test_obj.reset_index(drop=True)], axis=1)

    return X_train, X_test, encoder

def final_prediction(train_df, test_df, params, selected_features,
                    is_summer=True, target_col="heat_demand",
                    use_weights=True, weight_type='log1p', random_state=42):
    """
    최종 예측 함수

    Parameters:
    - train_df: 훈련 데이터
    - test_df: 테스트 데이터
    - params: XGBoost 하이퍼파라미터 딕셔너리
    - selected_features: 선택된 피처 리스트
    - is_summer: 여름 데이터 여부 (True: 여름, False: 비여름)
    - target_col: 타겟 컬럼명
    - use_weights: 가중치 사용 여부
    - weight_type: 가중치 계산 방법
    - random_state: 랜덤 시드

    Returns:
    - predictions: 예측값 배열
    - model: 학습된 모델
    - encoder: 인코더
    """

    print(f"최종 모델 학습 및 예측을 시작합니다...")
    print(f"시즌: {'여름' if is_summer else '비여름'}")
    print(f"가중치 사용: {use_weights} ({'가중치 방법: ' + weight_type if use_weights else ''})")
    print(f"선택된 피처 개수: {len(selected_features)}")

    # objective 설정
    objective = "reg:absoluteerror" if is_summer else "reg:squarederror"

    # 데이터 정렬
    train_df = train_df.copy().sort_values(["branch_id", "tm"]).reset_index(drop=True)

    # 피처 준비
    X_train, X_test, encoder = prepare_features(train_df, test_df, target_col)

    # 선택된 피처만 사용
    X_train_selected = X_train[selected_features]
    X_test_selected = X_test[selected_features]

    # 모델 파라미터 설정
    model_params = {
        'random_state': random_state,
        'n_jobs': -1,
        'tree_method': "gpu_hist",
        'objective': objective,
        **params
    }

    # 모델 생성 및 학습
    model = XGBRegressor(**model_params)

    # 가중치 적용하여 학습
    if use_weights:
        sample_weights = calculate_sample_weights(train_df[target_col], weight_type)
        model.fit(X_train_selected, train_df[target_col], sample_weight=sample_weights)
        print(f"가중치({weight_type})가 적용되었습니다.")
    else:
        model.fit(X_train_selected, train_df[target_col])

    # 예측
    predictions = model.predict(X_test_selected)

    print(f"예측 완료! 예측값 개수: {len(predictions)}")

    return predictions, model, encoder

In [ ]:
# 여름 파라미터와 피처
summer_params = {
    'n_estimators': 977,
    'max_depth': 10,
    'learning_rate': 0.014774640873115115,
    'subsample': 0.6629214558105696,
    'colsample_bytree': 0.6301411350439541,
    'reg_alpha': 3.353882208189202,
    'reg_lambda': 0.7115646469357197,
    'min_child_weight': 4,
    'gamma': 0.18694370838960597
    }


summer_features = ['branch_id_C', 'hour', 'hour_cos','si_roll3', 'si_lag22',
  'si_lag2', 'si_lag12','hour_of_week','day_flag','weekofyear',
  'branch_id_B','month','ta_chi_roll168','year','heatwave','branch_id_G','hour_sin',
  'DI_interaction','ta_chi_roll24','ta_roll24','ta_roll168','si_lag24',
  'si_lag3','si','si_lag1','si_lag15','si_lag14','ta_chi_roll72','ta_chi_roll3',
  'si_lag13','ta_chi','si_lag21','ta_roll72','rn_hr1_roll168','si_lag4',
  'monthday_cos','ws_roll168','hm_roll168','si_lag8','si_roll168','si_lag23',
  'dew_point','monthday_sin','rn_day_lag1','weekhour_cos','rn_hr1_roll24',
  'ws_roll72','rn_day','si_roll72','rn_hr1_roll72','hm_roll72','weekhour_sin',
  'ta_lag2','ws_roll24','si_lag16','ws_roll3','day','hm_roll24','si_lag11',
  'heating_degree','si_roll24','hm_lag1','si_lag20','rn_hr1','si_lag19',
  'ta_roll3','cumulative_si','ta_lag3','rn_hr1_roll3','si_lag17','ta_lag10',
  'si_lag10','ta_chi_lag1','si_lag18','wd_cos_lag2','hm_lag11','ta_lag1',
  'wd_sin','wd_cos','hm_roll3','cooling_degree','wd_cos_lag24'
  ]

# 여름 데이터 예측
summer_predictions, summer_model, summer_encoder = final_prediction(
    train_df=cluster1_summer_df,
    test_df=cluster1_summer_df_test,
    params=summer_params,
    selected_features=summer_features,
    is_summer=True,
    use_weights=True,
    weight_type='log1p'
)


최종 모델 학습 및 예측을 시작합니다...
시즌: 여름
가중치 사용: True (가중치 방법: log1p)
선택된 피처 개수: 82
가중치(log1p)가 적용되었습니다.
예측 완료! 예측값 개수: 8784


In [ ]:
# 비여름 파라미터와 피처
non_summer_params = {'n_estimators': 918,
  'max_depth': 5,
  'learning_rate': 0.043873642758915754,
  'subsample': 0.8150410596513636,
  'colsample_bytree': 0.749445591829446,
  'reg_alpha': 8.958074842675725,
  'reg_lambda': 0.8725166232765738,
  'min_child_weight': 9,
  'gamma': 4.375099492033362
                     }

non_summer_features = [
    'season_group_winter', 'ta_chi_roll72', 'ta_chi_roll168', 'season_group_mid_season',
    'heating_degree', 'ta_chi_roll24', 'ta_zone_low', 'ta_chi', 'ta', 'si_lag2', 'ta_chi_lag14',
    'branch_id_C', 'branch_id_G', 'hour', 'day_flag', 'si_lag1', 'ta_chi_lag17', 'si_roll3',
    'ta_chi_lag13', 'si_lag15', 'ta_chi_lag1', 'ta_lag15', 'ta_lag14', 'si_lag16',
    'ta_lag13', 'ta_roll72', 'si_lag17', 'weekofyear', 'ta_roll24', 'ta_chi_roll3',
    'si_lag7', 'year', 'ta_chi_lag16', 'si_lag10', 'si_lag11', 'hour_cos', 'ta_chi_lag3',
    'si_lag4', 'hour_of_week', 'ws_roll3', 'si_lag19', 'ta_lag16', 'hour_sin', 'si_lag18',
    'dew_point', 'ta_lag1', 'ws_lag5', 'ta_lag2', 'si_lag3', 'branch_id_B', 'rn_hr1_roll72',
    'ta_chi_lag6', 'ta_roll168', 'cumulative_si', 'si_roll72', 'ta_chi_lag2', 'ta_lag6',
    'month_sin', 'ta_zone_mid', 'ta_lag18', 'ws_roll168', 'ta_lag4', 'ta_chi_lag4',
    'season_group_early_winter', 'hm_roll168', 'si_lag14', 'si_lag5', 'ws_lag3',
    'rn_hr1_roll3', 'ta_roll3', 'day', 'ws_group_약풍', 'ws_lag7', 'si_roll168',
    'ta_chi_lag11', 'ws_lag9', 'ta_chi_lag7', 'month', 'rn_hr1_roll24', 'hm_lag8',
    'wd_cos_lag1', 'hm_lag14', 'hm_lag10', 'ta_lag7', 'rn_day_lag1', 'ta_chi_lag5',
    'ta_chi_lag22', 'monthday_cos', 'ta_lag5', 'ta_diff_chi', 'weekhour_cos',
    'monthday_sin', 'wd_cos_lag2', 'hm_lag24', 'ta_chi_lag21', 'month_cos', 'ta_chi_lag8',
    'si_roll24', 'ws_lag21', 'rn_day_lag2', 'wd_sin_lag5', 'ws_roll72', 'weekhour_sin',
    'si_lag6', 'hm_lag15', 'ta_chi_lag20', 'ws_roll24', 'rn_hr1_roll168', 'hm_lag4',
    'hm_lag11', 'ws_lag15', 'hm_roll72', 'wd_cos_lag7', 'ta_lag10', 'si']



# 비여름 데이터 예측
non_summer_predictions, non_summer_model, non_summer_encoder = final_prediction(
    train_df=cluster1_non_summmer_df,
    test_df=cluster1_non_summer_df_test,
    params=non_summer_params,
    selected_features=non_summer_features,
    is_summer=False,
    use_weights=True,
    weight_type='log1p'
)

최종 모델 학습 및 예측을 시작합니다...
시즌: 비여름
가중치 사용: True (가중치 방법: log1p)
선택된 피처 개수: 115
가중치(log1p)가 적용되었습니다.
예측 완료! 예측값 개수: 17571


In [ ]:
# 예측 결과 정리
summer_df = cluster1_summer_df_test.copy()
summer_df['heat_demand_pred'] = summer_predictions
summer_result_df = summer_df[['tm', 'branch_id', 'heat_demand_pred']].copy()

non_summer_df = cluster1_non_summer_df_test.copy()
non_summer_df['heat_demand_pred'] = non_summer_predictions
non_summer_result_df = non_summer_df[['tm', 'branch_id', 'heat_demand_pred']].copy()

# 최종 결과 통합
cluster1_result_df = pd.concat([summer_result_df, non_summer_result_df], axis=0).sort_values(['tm', 'branch_id']).reset_index(drop=True)
cluster1_result_df

,tm,branch_id,heat_demand_pred
0,2024-01-01 00:00:00,B,476.376343
1,2024-01-01 00:00:00,C,519.083496
2,2024-01-01 00:00:00,G,448.257904
3,2024-01-01 01:00:00,B,440.297913
4,2024-01-01 01:00:00,C,501.981079
...,...,...,...
26350,2024-12-31 23:00:00,C,632.655762
26351,2024-12-31 23:00:00,G,518.299438
26352,2025-01-01 00:00:00,B,559.363586
26353,2025-01-01 00:00:00,C,575.154663


#### (3) Cluster2

In [ ]:
def calculate_sample_weights(y_values, weight_type='log1p'):
    """
    샘플 가중치 계산 함수
    """
    y_values = np.array(y_values)

    if weight_type == 'log1p':
        weights = np.log1p(y_values)
    elif weight_type == 'linear':
        weights = y_values
    elif weight_type == 'sqrt':
        weights = np.sqrt(np.maximum(y_values, 0))
    elif weight_type == 'quadratic':
        weights = np.square(y_values)
    else:
        raise ValueError("weight_type must be one of: 'log1p', 'linear', 'sqrt', 'quadratic'")

    # 가중치 정규화 (평균을 1로)
    weights = weights / np.mean(weights)
    return weights

def prepare_features(train_df, test_df, target_col, encoder=None):
    """피처 준비 및 인코딩"""
    # 범주형 변수 원핫 인코딩
    object_cols = train_df.select_dtypes(include="object").columns.tolist()

    if encoder is None:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        if object_cols:
            encoder.fit(train_df[object_cols])

    # 훈련 데이터 피처 준비
    if object_cols:
        X_train_obj = pd.DataFrame(
            encoder.transform(train_df[object_cols]),
            columns=encoder.get_feature_names_out(object_cols)
        )
        X_test_obj = pd.DataFrame(
            encoder.transform(test_df[object_cols]),
            columns=encoder.get_feature_names_out(object_cols)
        )
    else:
        X_train_obj = pd.DataFrame()
        X_test_obj = pd.DataFrame()

    # 수치형 변수
    exclude_cols = object_cols + [target_col, "tm", "heat_demand_log"]
    X_train_num = train_df.drop(columns=exclude_cols, errors='ignore')
    X_test_num = test_df.drop(columns=exclude_cols, errors='ignore')

    # 피처 결합
    X_train = pd.concat([X_train_num.reset_index(drop=True), X_train_obj.reset_index(drop=True)], axis=1)
    X_test = pd.concat([X_test_num.reset_index(drop=True), X_test_obj.reset_index(drop=True)], axis=1)

    return X_train, X_test, encoder

def final_prediction(train_df, test_df, params, selected_features,
                    is_summer=True, target_col="heat_demand",
                    use_weights=True, weight_type='log1p', random_state=42):
    """
    최종 예측 함수

    Parameters:
    - train_df: 훈련 데이터
    - test_df: 테스트 데이터
    - params: XGBoost 하이퍼파라미터 딕셔너리
    - selected_features: 선택된 피처 리스트
    - is_summer: 여름 데이터 여부 (True: 여름, False: 비여름)
    - target_col: 타겟 컬럼명
    - use_weights: 가중치 사용 여부
    - weight_type: 가중치 계산 방법
    - random_state: 랜덤 시드

    Returns:
    - predictions: 예측값 배열
    - model: 학습된 모델
    - encoder: 인코더
    """

    print(f"최종 모델 학습 및 예측을 시작합니다...")
    print(f"시즌: {'여름' if is_summer else '비여름'}")
    print(f"가중치 사용: {use_weights} ({'가중치 방법: ' + weight_type if use_weights else ''})")
    print(f"선택된 피처 개수: {len(selected_features)}")

    # objective 설정
    objective = "reg:absoluteerror" if is_summer else "reg:squarederror"

    # 데이터 정렬
    train_df = train_df.copy().sort_values(["branch_id", "tm"]).reset_index(drop=True)

    # 피처 준비
    X_train, X_test, encoder = prepare_features(train_df, test_df, target_col)

    # 선택된 피처만 사용
    X_train_selected = X_train[selected_features]
    X_test_selected = X_test[selected_features]

    # 모델 파라미터 설정
    model_params = {
        'random_state': random_state,
        'n_jobs': -1,
        'tree_method': "gpu_hist",
        'objective': objective,
        **params
    }

    # 모델 생성 및 학습
    model = XGBRegressor(**model_params)

    # 가중치 적용하여 학습
    if use_weights:
        sample_weights = calculate_sample_weights(train_df[target_col], weight_type)
        model.fit(X_train_selected, train_df[target_col], sample_weight=sample_weights)
        print(f"가중치({weight_type})가 적용되었습니다.")
    else:
        model.fit(X_train_selected, train_df[target_col])

    # 예측
    predictions = model.predict(X_test_selected)

    print(f"예측 완료! 예측값 개수: {len(predictions)}")

    return predictions, model, encoder

In [ ]:
# 여름 파라미터와 피처
summer_params = {
    'n_estimators': 917,
    'max_depth': 6,
    'learning_rate': 0.015542606028500787,
    'subsample': 0.6531672620804118,
    'colsample_bytree': 0.757275749281304,
    'reg_alpha': 9.999931437682223,
    'reg_lambda': 7.240382931891751,
    'min_child_weight': 10,
    'gamma': 3.3369443328815587
}

summer_features = summer_features = [
    'si_lag1', 'DI_interaction', 'hour_cos', 'day_flag', 'branch_id_A', 'hour', 'branch_id_D',
    'hour_of_week', 'hour_sin', 'branch_id_P', 'weekofyear', 'si_lag23', 'si_lag21', 'month',
    'si', 'ta_chi_roll168', 'si_lag22', 'year', 'ta_chi_lag23', 'ta_chi_roll72', 'si_roll168',
    'branch_id_H', 'si_lag2', 'heatwave', 'monthday_cos', 'si_roll3', 'si_lag20', 'ta_roll168',
    'ta_roll24', 'si_lag19', 'ta_diff_chi', 'ta_chi_roll24', 'si_lag3', 'hm_roll3', 'hm_roll72',
    'hm_lag13', 'ta_chi', 'cumulative_si', 'weekhour_cos', 'hm_lag10', 'hm_lag7', 'si_lag24',
    'ws_roll168', 'dew_point', 'rn_day_lag16', 'si_roll24', 'rn_hr1_roll168', 'hm_roll168',
    'cooling_degree', 'rn_day_lag12', 'monthday_sin', 'weekhour_sin', 'ta_lag11', 'rn_hr1_roll24',
    'rn_hr1_roll72', 'ws_roll72', 'day', 'hm_lag21', 'rn_day_lag20', 'hm_roll24', 'rn_day_lag11',
    'rn_day_lag19', 'hm_lag24', 'heating_degree', 'rn_hr1_roll3', 'rn_day_lag21', 'si_roll72',
    'ws_roll24', 'si_lag4', 'ta_roll72', 'rn_hr1', 'ta_chi_lag3', 'ta_chi_lag24', 'ta_lag3',
    'rn_day_lag23', 'si_lag18', 'hm_lag16', 'rn_day_lag14', 'hm_lag8', 'rn_day_lag24', 'hm_lag14',
    'hm_lag15', 'ta_lag10', 'ta_chi_lag1', 'ws_roll3', 'ta_chi_lag2', 'rn_day', 'hm_lag9',
    'rn_day_lag17', 'rn_hr1_lag23', 'hm_lag2', 'hm_lag12', 'ta_lag9', 'ta_roll3', 'hm',
    'ta_chi_roll3', 'hm_lag3', 'hm_lag1', 'hm_lag20', 'hm_lag17', 'ta', 'hm_lag22', 'wd_sin',
    'hm_lag11', 'hm_lag23', 'rn_day_lag10', 'ta_lag2', 'rn_day_lag22', 'rn_hr1_lag24', 'ws_group_약풍'
]

# 여름 데이터 예측
summer_predictions, summer_model, summer_encoder = final_prediction(
    train_df=cluster2_summer_df,
    test_df=cluster2_summer_df_test,
    params=summer_params,
    selected_features=summer_features,
    is_summer=True,
    use_weights=True,
    weight_type='log1p'
)


최종 모델 학습 및 예측을 시작합니다...
시즌: 여름
가중치 사용: True (가중치 방법: log1p)
선택된 피처 개수: 110
가중치(log1p)가 적용되었습니다.
예측 완료! 예측값 개수: 11712


In [ ]:
# 비여름 파라미터와 피처
non_summer_params = {
    'n_estimators': 789,
    'max_depth': 4,
    'learning_rate': 0.07828782968379985,
    'subsample': 0.6256021918603922,
    'colsample_bytree': 0.6786724475759028,
    'reg_alpha': 9.82692246895845,
    'reg_lambda': 6.407730512882728,
    'min_child_weight': 1,
    'gamma': 3.508478999047011
}

non_summer_features = [
    'season_group_winter', 'ta_chi_roll168', 'season_group_early_winter', 'ta_chi_roll72',
    'heating_degree', 'ta_chi_roll24', 'branch_id_P', 'branch_id_D', 'ta_chi_roll3', 'ta_chi_lag1',
    'branch_id_A', 'day_flag', 'ta_zone_low', 'season_group_mid_season', 'ta_lag1', 'ta_chi',
    'ta_chi_lag15', 'branch_id_H', 'si_lag15', 'ta_chi_lag13', 'si_roll3', 'ta_lag2', 'ta_lag12',
    'hour', 'ta', 'si_lag7', 'si_lag1', 'si_lag2', 'si_lag14', 'ta_chi_lag2', 'ta_roll168',
    'weekofyear', 'si_lag9', 'ta_roll72', 'ta_chi_lag12', 'ta_chi_lag3', 'year', 'si_lag3',
    'ta_chi_lag4', 'si_lag21', 'si_lag16', 'month', 'si_lag20', 'hour_of_week', 'si', 'ta_lag17',
    'month_sin', 'rn_day_lag19', 'si_lag17', 'si_lag4', 'hour_sin', 'month_cos', 'ta_chi_lag6',
    'hour_cos', 'ta_lag14', 'rn_hr1_roll72', 'rn_hr1_roll24', 'ta_lag15', 'si_roll168',
    'ta_chi_lag8', 'si_lag18', 'si_lag13', 'ta_chi_lag9', 'ta_chi_lag11', 'rn_day_lag14',
    'ta_roll24', 'ta_chi_lag20', 'rn_day', 'ta_chi_lag24', 'hm_roll168', 'si_roll72', 'ta_lag24',
    'wd_sin_lag7', 'si_roll24', 'ta_diff_chi', 'rn_day_lag10', 'ta_chi_lag17', 'si_lag5',
    'ws_lag9', 'ta_lag3', 'day', 'si_lag19', 'dew_point', 'ta_chi_lag18', 'hm', 'ws_roll3',
    'wd_lag1', 'ta_lag16', 'rn_day_lag11', 'cooling_degree', 'cumulative_si', 'wd_cos_lag7',
    'ta_zone_mid', 'ws_roll168', 'ta_lag6', 'ta_chi_lag7', 'ta_chi_lag16', 'ta_lag7', 'si_lag11',
    'monthday_sin', 'rn_day_lag16', 'ta_lag13', 'rn_day_lag13', 'monthday_cos', 'hm_roll24',
    'ws_roll72', 'rn_hr1_roll168', 'ta_chi_lag14', 'si_lag6', 'ta_chi_lag19', 'weekhour_cos',
    'rn_day_lag12', 'ta_lag10', 'hm_lag14', 'hm_lag17', 'hm_lag11', 'ta_chi_lag21', 'ta_chi_lag22',
    'ws_lag16', 'wd_lag8', 'hm_roll72', 'wd_sin_lag8', 'hm_roll3', 'ws_lag5', 'wd_rad',
    'rn_day_lag17', 'ta_lag18', 'wd_sin_lag14', 'ta_chi_lag10', 'hm_lag8', 'hm_lag5', 'ws_roll24',
    'ta_lag8', 'rn_hr1_lag23', 'hm_lag9', 'wd_cos_lag1', 'hm_lag10', 'hm_lag7', 'wd_sin_lag9',
    'wd_cos_lag15', 'rn_hr1_lag1', 'ta_chi_lag23', 'wd_cos', 'rn_hr1_lag5', 'ta_chi_lag5',
    'hm_lag12', 'wd_cos_lag11', 'wd_cos_lag8', 'hm_lag24', 'hm_lag13', 'ta_lag23', 'rn_hr1',
    'ta_lag22', 'rn_hr1_lag4'
]

# 비여름 데이터 예측
non_summer_predictions, non_summer_model, non_summer_encoder = final_prediction(
    train_df=cluster2_non_summmer_df,
    test_df=cluster2_non_summer_df_test,
    params=non_summer_params,
    selected_features=non_summer_features,
    is_summer=False,
    use_weights=True,
    weight_type='log1p'
)

최종 모델 학습 및 예측을 시작합니다...
시즌: 비여름
가중치 사용: True (가중치 방법: log1p)
선택된 피처 개수: 154
가중치(log1p)가 적용되었습니다.
예측 완료! 예측값 개수: 23428


In [ ]:
# 예측 결과 정리
summer_df = cluster2_summer_df_test.copy()
summer_df['heat_demand_pred'] = summer_predictions
summer_result_df = summer_df[['tm', 'branch_id', 'heat_demand_pred']].copy()

non_summer_df = cluster2_non_summer_df_test.copy()
non_summer_df['heat_demand_pred'] = non_summer_predictions
non_summer_result_df = non_summer_df[['tm', 'branch_id', 'heat_demand_pred']].copy()

# 최종 결과 통합
cluster2_result_df = pd.concat([summer_result_df, non_summer_result_df], axis=0).sort_values(['tm', 'branch_id']).reset_index(drop=True)
cluster2_result_df

,tm,branch_id,heat_demand_pred
0,2024-01-01 00:00:00,A,221.422577
1,2024-01-01 00:00:00,D,347.781372
2,2024-01-01 00:00:00,H,325.554047
3,2024-01-01 00:00:00,P,200.630875
4,2024-01-01 01:00:00,A,216.449997
...,...,...,...
35135,2024-12-31 23:00:00,P,213.962280
35136,2025-01-01 00:00:00,A,248.612762
35137,2025-01-01 00:00:00,D,371.616699
35138,2025-01-01 00:00:00,H,357.959137


#### (4) Cluster3

In [ ]:
top_features1 = [
    'season_group_winter', 'season_group_early_winter', 'branch_id_M', 'branch_id_L',
    'ta_chi_roll168', 'heating_degree', 'ta_chi', 'si_lag9', 'si_lag1', 'ta_lag14',
    'ta_chi_roll72', 'branch_id_R', 'hour', 'ta_chi_lag15', 'day_flag',
    'temp_humidity_interaction', 'ta_chi_lag14', 'ta_lag1', 'ta', 'year'
]

In [ ]:
#데이터준비
df_cluster3 = cluster3_df.copy()
df_cluster3['tm'] = pd.to_datetime(df_cluster3['tm'])
df_cluster3 = df_cluster3.sort_values(["tm", "branch_id"]).reset_index(drop=True)

# 타겟 지정
target_col = "heat_demand"
y = df_cluster3[target_col]

# object 컬럼 원핫 인코딩
object_cols = df_cluster3.select_dtypes(include="object").columns.tolist()
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded_array = encoder.fit_transform(df_cluster3[object_cols])
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(object_cols))

# 나머지 수치형 변수
exclude_cols = object_cols + [target_col, "heat_demand", "heat_demand_log"]
numeric_df = df_cluster3.drop(columns=exclude_cols)

# 최종 feature
X = pd.concat([numeric_df.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)

#cluster_3에만 추가한 온도,습도 상호작용항
X['temp_humidity_interaction'] = X['ta'] * X['hm']

X['tm'] = pd.to_datetime(X['tm'])  # 만약 이미 datetime이면 이 줄 생략

# 1시간 단위로 시간 블록 생성 (연-월-일-시까지 포함됨)
X['time_block'] = X['tm'].dt.strftime('%Y-%m-%d %H:00:00')

for col in top_features1:
    new_col = f'{col}_squared'
    X[new_col] = X[col] ** 2

X = X.drop('tm',axis=1)

# 고유한 시간 블록 추출 및 정렬
time_blocks = np.sort(X['time_block'].unique())

In [ ]:
top_features2 = [
    'season_group_winter', 'season_group_early_winter', 'branch_id_M', 'branch_id_L',
    'heating_degree', 'ta_chi_roll168', 'ta_chi', 'si_lag9', 'ta_lag14', 'si_lag1',
    'ta_chi_roll72_squared', 'branch_id_R', 'ta_chi_roll72', 'ta_chi_lag15', 'hour',
    'day_flag', 'ta_chi_lag1', 'year', 'ta_chi_lag14', 'ta_chi_squared',
    'ta_lag1', 'ta_chi_lag13', 'ta_roll72', 'ta', 'branch_id_S',
    'ta_chi_lag14_squared', 'ta_roll168', 'si_lag2', 'temp_humidity_interaction',
    'ta_chi_lag18', 'coldwave', 'si_roll3', 'ta_roll24', 'si_lag15',
    'ta_chi_lag17', 'ta_zone_low', 'ta_chi_lag6', 'DI_interaction', 'weekofyear',
    'ta_chi_roll24', 'ta_lag11', 'hour_cos', 'ta_squared', 'ta_lag2',
    'ta_lag24', 'si', 'ta_chi_lag2', 'month', 'ta_lag13',
    'month_cos', 'ta_chi_lag3', 'hour_of_week', 'hour_sin', 'ta_lag25',
    'si_lag19', 'rn_day_lag9', 'ta_lag23', 'ta_chi_lag9', 'ta_chi_lag12',
    'dew_point', 'cumulative_si', 'si_lag18', 'si_lag5', 'ta_lag7'
]

In [ ]:
X = X[top_features2]

best_params =  {
    'n_estimators': 602,
    'max_depth': 7,
    'learning_rate': 0.03878518685897426,
    'subsample': 0.8613128362703897,
    'colsample_bytree': 0.6347674724905161,
    'reg_alpha': 9.545821121497825,
    'reg_lambda': 1.258672944425539e-06
}

sample_weight = np.log1p(np.expm1(y)) #y, 선형 가중치

# 스케일러 객체 생성
scaler = StandardScaler()
# fit_transform
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

model_final = XGBRegressor(random_state=42, n_jobs=-1, tree_method="gpu_hist", predictor="gpu_predictor" ,**best_params)
model_final.fit(X, y,sample_weight=sample_weight)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6347674724905161, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03878518685897426, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=602, n_jobs=-1,
             num_parallel_tree=None, predictor='gpu_predictor', ...)

In [ ]:
# 데이터 준비
cluster3_test_df = cluster3_df_test.copy()

# test 데이터 정렬 및 인덱스 재설정
df_test_cluster3 = cluster3_test_df.sort_values(["tm", "branch_id"]).reset_index(drop=True)

# test 데이터에서 object 컬럼 추출
object_cols_test = df_test_cluster3.select_dtypes(include="object").columns.tolist()

# train 때 사용한 encoder를 그대로 쓰기 때문에 여기서는 transform만 한다
encoded_array_test = encoder.transform(df_test_cluster3[object_cols_test])
encoded_df_test = pd.DataFrame(encoded_array_test, columns=encoder.get_feature_names_out(object_cols_test))

# 수치형 변수 추출
exclude_cols_test = object_cols_test + ["tm", "heat_demand","heat_demand_log"]  # test에는 타겟 없음
numeric_df_test = df_test_cluster3.drop(columns=exclude_cols_test)

# 최종 test 피처 구성
X_test = pd.concat([numeric_df_test.reset_index(drop=True), encoded_df_test.reset_index(drop=True)], axis=1)

#cluster_3에만 추가한 온도,습도 상호작용항
X_test['temp_humidity_interaction'] = X_test['ta'] * X_test['hm']

#선택된 변수 제곱 파생변수 추가
for col in top_features1:
    new_col = f'{col}_squared'
    X_test[new_col] = X_test[col] ** 2

#2번째 선택된 변수로 축소
X_test = X_test[top_features2]

# transform
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# test 데이터 예측
y_test_pred= model_final.predict(X_test)

# 결과 확인 (예: test 데이터에 예측 칼럼 추가)
df_test_cluster3['heat_demand_pred'] = y_test_pred

df_test_cluster3 = df_test_cluster3[['tm', 'branch_id', 'heat_demand_pred']]

# **3. 최종 결과**

In [ ]:
test_for_submission = pd.read_csv('/content/drive/MyDrive/체크난방/data/test_heat.csv')

In [ ]:
test_for_submission['TM'] = test_for_submission['TM'].astype(str)

In [ ]:
def apply_cluster_result_merge(df, test, verbose=True):
    df_temp = df.copy()
    df_temp['tm'] = pd.to_datetime(df_temp['tm'])
    df_temp['TM'] = df_temp['tm'].dt.strftime('%Y%m%d%H')  # str type
    df_temp['branch_ID'] = df_temp['branch_id'].astype(str).str.upper().str.strip()

    # 강제 형변환으로 형 일치
    test['TM'] = test['TM'].astype(str)
    test['branch_ID'] = test['branch_ID'].astype(str).str.upper().str.strip()

    # 매칭 키 확인
    df_keys = set(zip(df_temp['TM'], df_temp['branch_ID']))
    test_keys = set(zip(test['TM'], test['branch_ID']))
    matched_keys = df_keys & test_keys

    # merge
    merged = test.merge(
        df_temp[['TM', 'branch_ID', 'heat_demand_pred']],
        on=['TM', 'branch_ID'], how='left'
    )

    merged['heat_demand'] = merged['heat_demand_pred'].combine_first(merged['heat_demand'])
    merged['heat_demand'] = merged['heat_demand'].round(1)
    merged = merged.drop(columns=['heat_demand_pred'])

    return merged

In [ ]:
test_for_submission = apply_cluster_result_merge(df_test_cluster0, test_for_submission)
test_for_submission = apply_cluster_result_merge(cluster1_result_df, test_for_submission)
test_for_submission = apply_cluster_result_merge(cluster2_result_df, test_for_submission)
test_for_submission = apply_cluster_result_merge(df_test_cluster3, test_for_submission)
test_for_submission

,TM,branch_ID,TA,WD,WS,RN_DAY,RN_HR1,HM,SI,ta_chi,heat_demand
0,2024010100,A,0.5,171.3,0.8,2.5,0.0,97.1,-99.0,0.3,221.4
1,2024010101,A,0.4,93.7,1.0,0.0,0.0,96.8,-99.0,0.1,216.4
2,2024010102,A,-0.1,133.0,0.8,0.0,0.0,97.0,-99.0,0.0,202.8
3,2024010103,A,-0.8,218.6,0.6,0.0,0.0,96.9,-99.0,-0.2,206.1
4,2024010104,A,0.1,58.7,1.5,0.0,0.0,97.0,-99.0,-0.1,207.1
...,...,...,...,...,...,...,...,...,...,...,...
166910,2024123120,S,-1.1,360.0,0.0,0.0,0.0,45.8,-99.0,-1.7,29.1
166911,2024123121,S,-1.3,360.0,0.0,0.0,0.0,48.3,-99.0,-2.3,31.0
166912,2024123122,S,-2.4,360.0,0.0,0.0,0.0,60.0,-99.0,-3.1,32.1
166913,2024123123,S,-3.6,360.0,0.0,0.0,0.0,65.7,-99.0,-3.9,31.5


In [ ]:
test_for_submission.to_csv('/content/drive/MyDrive/체크난방/0627-1.csv')